# 02 — Schema-driven Silver formatter

This version treats `schema_definition.csv` as a contract. It generates and
executes DDL before loading data, emits every contract column in ordinal order,
logs missing/extra columns, and formats both latest Bronze and archived Bronze.

- Latest source → `silver.slv_<source_schema>_<table>`
- Archive source → `silver_archive.slv_<source_schema>_<table>`

Archive data is deliberately kept in a separate Silver schema to avoid mixing
historical exports with the current snapshot and double counting records.

In [11]:
BRONZE_SCHEMA = "bronze"
ARCHIVE_SCHEMA = "archived"
SILVER_SCHEMA = "silver"
SILVER_ARCHIVE_SCHEMA = "silver_archive"
SCHEMA_CSV_PATH = "Files/cfg_files/schema_definition.csv"
LATEST_PREFIXES = ("brz_",)
ARCHIVE_PREFIXES = ("archived_",)
REBUILD = False
STRICT_SCHEMA = True
FAIL_ON_TABLE_ERROR = True
ARCHIVE_ENFORCE_NOT_NULL = False  # older exports may pre-date required columns; drift is logged
DATE_FORMATS = ["yyyy-MM-dd", "dd/MM/yyyy", "yyyy-MM-dd'T'HH:mm:ss"]
TIMESTAMP_FORMATS = ["yyyy-MM-dd HH:mm:ss", "yyyy-MM-dd'T'HH:mm:ss", "yyyy-MM-dd'T'HH:mm:ss.SSSXXX"]

StatementMeta(, 43152dc0-bcee-41bc-8411-b4bfc63e5f32, 13, Finished, Available, Finished, False)

In [12]:
import re, uuid
from collections import defaultdict
from datetime import datetime
from pyspark.sql import functions as F

RUN_ID = str(uuid.uuid4())
STARTED_AT = datetime.utcnow()

def qident(value):
    return "`" + str(value).replace("`", "``") + "`"

def normalise(value):
    return re.sub(r"[^a-z0-9]", "", (value or "").lower())

def append_rows(table_name, rows, schema):
    if rows:
        spark.createDataFrame(rows, schema).write.format("delta").mode("append").saveAsTable(table_name)

StatementMeta(, 43152dc0-bcee-41bc-8411-b4bfc63e5f32, 14, Finished, Available, Finished, False)

In [13]:
spark.sql("CREATE SCHEMA IF NOT EXISTS monitoring")
spark.sql("""
CREATE TABLE IF NOT EXISTS monitoring.cfg_pipeline_run (
  run_id STRING, pipeline_name STRING, layer STRING, source_kind STRING,
  started_at TIMESTAMP, ended_at TIMESTAMP, status STRING,
  tables_succeeded INT, tables_failed INT, rows_read BIGINT, rows_written BIGINT,
  error_message STRING
) USING DELTA
""")
spark.sql("""
CREATE TABLE IF NOT EXISTS monitoring.cfg_table_load_metric (
  run_id STRING, layer STRING, source_kind STRING, source_object STRING,
  target_object STRING, rows_read BIGINT, rows_written BIGINT,
  duplicate_key_count BIGINT, null_primary_key_count BIGINT,
  recorded_at TIMESTAMP
) USING DELTA
""")

for schema_name in (SILVER_SCHEMA):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")
spark.sql("""
CREATE TABLE IF NOT EXISTS monitoring.cfg_silver_load_control (
  run_id STRING, source_kind STRING, source_table STRING, target_table STRING,
  contract_schema STRING, contract_table STRING, status STRING,
  rows_read BIGINT, rows_written BIGINT, missing_column_count INT,
  extra_column_count INT, started_at TIMESTAMP, ended_at TIMESTAMP,
  error_message STRING
) USING DELTA
""")
spark.sql("""
CREATE TABLE IF NOT EXISTS monitoring.cfg_schema_drift_event (
  run_id STRING, source_kind STRING, source_table STRING, target_table STRING,
  drift_type STRING, column_name STRING, expected_type STRING,
  detected_at TIMESTAMP
) USING DELTA
""")
append_rows("monitoring.cfg_pipeline_run", [(RUN_ID, "02_silver_formatter", "SILVER", "LATEST+ARCHIVE",
    STARTED_AT, None, "RUNNING", 0, 0, 0, 0, None)],
    "run_id string,pipeline_name string,layer string,source_kind string,started_at timestamp,ended_at timestamp,status string,tables_succeeded int,tables_failed int,rows_read long,rows_written long,error_message string")

StatementMeta(, 43152dc0-bcee-41bc-8411-b4bfc63e5f32, 15, Finished, Available, Finished, False)

In [14]:
required_schema_columns = {
    "schema_name", "table_name", "ordinal_position", "column_name", "data_type",
    "is_nullable", "is_primary_key", "referenced_schema", "referenced_table", "referenced_column"
}
schema_df = (spark.read.format("csv").option("header", "true").option("quote", '"')
    .option("escape", '"').option("multiLine", "true").load(SCHEMA_CSV_PATH))
missing_metadata_columns = required_schema_columns - set(schema_df.columns)
if missing_metadata_columns:
    raise ValueError(f"schema_definition.csv is missing: {sorted(missing_metadata_columns)}")

schema_rows = [row.asDict(recursive=True) for row in schema_df.collect()]
schema_df.withColumn("contract_loaded_at", F.current_timestamp()) \
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("monitoring.cfg_schema_contract_column")
contracts = defaultdict(list)
for row in schema_rows:
    if row.get("schema_name") and row.get("table_name") and row.get("column_name"):
        row["ordinal_position"] = int(row.get("ordinal_position") or 999999)
        contracts[(row["schema_name"].lower(), row["table_name"].lower())].append(row)
for key in contracts:
    contracts[key].sort(key=lambda item: item["ordinal_position"])

contracts_by_table = defaultdict(list)
for key in contracts:
    contracts_by_table[normalise(key[1])].append(key)

print(f"Loaded {len(schema_rows):,} column definitions for {len(contracts):,} tables")

StatementMeta(, 43152dc0-bcee-41bc-8411-b4bfc63e5f32, 16, Finished, Available, Finished, False)

Loaded 651 column definitions for 54 tables


In [15]:
def map_data_type(pg_type):
    value = (pg_type or "").lower().strip()
    if "[]" in value:
        return "ARRAY<STRING>"
    if "uuid" in value or "json" in value or "text" in value or "character" in value or "varchar" in value:
        return "STRING"
    if value in {"smallint", "int2", "integer", "int", "int4"}:
        return "INT"
    if value in {"bigint", "int8"}:
        return "BIGINT"
    match = re.search(r"(?:numeric|decimal)\s*\((\d+)\s*,\s*(\d+)\)", value)
    if match:
        precision = min(int(match.group(1)), 38)
        scale = min(int(match.group(2)), precision)
        return f"DECIMAL({precision},{scale})"
    if "numeric" in value or "decimal" in value:
        return "DECIMAL(38,18)"
    if "double" in value or "float" in value or "real" in value:
        return "DOUBLE"
    if "boolean" in value or value == "bool":
        return "BOOLEAN"
    if value == "date":
        return "DATE"
    if "timestamp" in value:
        return "TIMESTAMP"
    if "time" in value:
        return "STRING"
    return "STRING"

LINEAGE_COLUMNS = [
    ("_record_source", "STRING"), ("_source_table", "STRING"),
    ("_source_file_path", "STRING"), ("_export_date", "DATE"),
    ("_silver_run_id", "STRING"), ("_silver_load_ts", "TIMESTAMP")
]

def build_create_table_ddl(source_schema, table_name, schema_cols, target_schema, target_table, enforce_nullability=True):
    column_ddls = [f"{qident(c['column_name'])} {map_data_type(c['data_type'])}"
                   + (" NOT NULL" if enforce_nullability and (c.get("is_nullable") or "YES").upper() == "NO" else "")
                   for c in schema_cols]
    column_ddls.extend(f"{qident(name)} {data_type}" for name, data_type in LINEAGE_COLUMNS)
    return (f"CREATE TABLE IF NOT EXISTS {qident(target_schema)}.{qident(target_table)} (\n  "
            + ",\n  ".join(column_ddls) + "\n) USING DELTA")

def resolve_contract(physical_table, prefixes):
    base = physical_table.lower()
    for prefix in prefixes:
        if base.startswith(prefix):
            base = base[len(prefix):]
    direct = contracts_by_table.get(normalise(base), [])
    if len(direct) == 1:
        return direct[0]
    composite = [key for key in contracts if normalise(key[0] + "_" + key[1]) == normalise(base)]
    if len(composite) == 1:
        return composite[0]
    if len(direct) > 1:
        raise ValueError(f"Ambiguous contract for {physical_table}: {direct}; rename source as <schema>__<table>")
    return None

def first_parsed(column, formats, parser):
    return F.coalesce(*[parser(column, fmt) for fmt in formats])

def cast_column(frame, definition):
    name = definition["column_name"]
    spark_type = map_data_type(definition["data_type"])
    if name not in frame.columns:
        return F.lit(None).cast(spark_type).alias(name)
    source = F.col(qident(name))
    if spark_type == "BOOLEAN":
        clean = F.lower(F.trim(source.cast("string")))
        return (F.when(clean.isin("true", "t", "1", "yes", "y"), F.lit(True))
            .when(clean.isin("false", "f", "0", "no", "n"), F.lit(False))
            .otherwise(F.lit(None).cast("boolean")).alias(name))
    if spark_type == "DATE":
        return first_parsed(source.cast("string"), DATE_FORMATS, F.to_date).alias(name)
    if spark_type == "TIMESTAMP":
        return first_parsed(source.cast("string"), TIMESTAMP_FORMATS, F.to_timestamp).alias(name)
    if spark_type.startswith("DECIMAL") or spark_type in {"INT", "BIGINT", "DOUBLE"}:
        return F.regexp_replace(source.cast("string"), r"[^0-9eE+\.\-]", "").cast(spark_type).alias(name)
    if spark_type == "ARRAY<STRING>":
        return F.when(source.isNull(), F.lit(None).cast("array<string>")) \
            .otherwise(F.split(F.regexp_replace(source.cast("string"), r"^[\{\[]|[\}\]]$", ""), r"\s*,\s*")).alias(name)
    return F.trim(source.cast("string")).alias(name)

def format_frame(frame, schema_cols, source_kind, source_table):
    expressions = [cast_column(frame, definition) for definition in schema_cols]
    source_file = F.col("_source_file_path") if "_source_file_path" in frame.columns else F.lit(None)
    export_date = F.col("_export_date").cast("date") if "_export_date" in frame.columns else F.lit(None).cast("date")
    return frame.select(*expressions, source_file.alias("_source_file_path"), export_date.alias("_export_date")) \
        .withColumn("_record_source", F.lit(source_kind)) \
        .withColumn("_source_table", F.lit(source_table)) \
        .withColumn("_silver_run_id", F.lit(RUN_ID)) \
        .withColumn("_silver_load_ts", F.current_timestamp())

StatementMeta(, 43152dc0-bcee-41bc-8411-b4bfc63e5f32, 17, Finished, Available, Finished, False)

In [16]:
source_sets = [
    {"source_schema": BRONZE_SCHEMA, "source_kind": "LATEST", "prefixes": LATEST_PREFIXES, "target_schema": SILVER_SCHEMA, "include_archive": False},
    {"source_schema": ARCHIVE_SCHEMA, "source_kind": "ARCHIVE", "prefixes": ARCHIVE_PREFIXES, "target_schema": SILVER_ARCHIVE_SCHEMA, "include_archive": True},
]
control_schema = "run_id string,source_kind string,source_table string,target_table string,contract_schema string,contract_table string,status string,rows_read long,rows_written long,missing_column_count int,extra_column_count int,started_at timestamp,ended_at timestamp,error_message string"
drift_schema = "run_id string,source_kind string,source_table string,target_table string,drift_type string,column_name string,expected_type string,detected_at timestamp"
metric_schema = "run_id string,layer string,source_kind string,source_object string,target_object string,rows_read long,rows_written long,duplicate_key_count long,null_primary_key_count long,recorded_at timestamp"
ok = failed = total_read = total_written = 0
errors = []

for source_set in source_sets:
    source_schema = source_set["source_schema"]
    source_kind = source_set["source_kind"]
    table_rows = spark.sql(f"SHOW TABLES IN {source_schema}").collect()
    physical_tables = [row.tableName for row in table_rows if not row.isTemporary]
    for physical_table in sorted(physical_tables):
        source_table = f"{source_schema}.{physical_table}"
        started = datetime.utcnow()
        target_table = None
        contract_key = None
        try:
            contract_key = resolve_contract(physical_table, source_set["prefixes"])
            if contract_key is None:
                message = f"No schema contract for {source_table}"
                if STRICT_SCHEMA:
                    raise ValueError(message)
                print("WARN", message)
                continue
            contract_schema, contract_table = contract_key
            schema_cols = contracts[contract_key]
            #target_name = f"slv_{contract_schema}_{contract_table}"
            target_name = f"slv_{contract_table}"
            target_table = f"{source_set['target_schema']}.{target_name}"
            frame = spark.table(source_table)
            source_count = frame.count()
            contract_columns = {c["column_name"] for c in schema_cols}
            technical_columns = {c for c in frame.columns if c.startswith("_")}
            missing = sorted(contract_columns - set(frame.columns))
            extra = sorted(set(frame.columns) - contract_columns - technical_columns)

            drift_rows = [(RUN_ID, source_kind, source_table, target_table, "MISSING", name,
                map_data_type(next(c["data_type"] for c in schema_cols if c["column_name"] == name)), datetime.utcnow()) for name in missing]
            drift_rows += [(RUN_ID, source_kind, source_table, target_table, "EXTRA", name, None,
                datetime.utcnow()) for name in extra]
            append_rows("monitoring.cfg_schema_drift_event", drift_rows, drift_schema)

            if REBUILD:
                spark.sql(f"DROP TABLE IF EXISTS {qident(source_set['target_schema'])}.{qident(target_name)}")
            ddl = build_create_table_ddl(contract_schema, contract_table, schema_cols,
                source_set["target_schema"], target_name,
                enforce_nullability=(source_kind == "LATEST" or ARCHIVE_ENFORCE_NOT_NULL))
            spark.sql(ddl)
            formatted = format_frame(frame, schema_cols, source_kind, source_table)
            spark.sql(f"TRUNCATE TABLE {qident(source_set['target_schema'])}.{qident(target_name)}")
            formatted.write.format("delta").mode("append").saveAsTable(target_table)
            written = formatted.count()

            append_rows("monitoring.cfg_silver_load_control", [(RUN_ID, source_kind, source_table, target_table,
                contract_schema, contract_table, "SUCCESS", source_count, written, len(missing), len(extra),
                started, datetime.utcnow(), None)], control_schema)
            append_rows("monitoring.cfg_table_load_metric", [(RUN_ID, "SILVER", source_kind, source_table,
                target_table, source_count, written, None, None, datetime.utcnow())], metric_schema)
            ok += 1; total_read += source_count; total_written += written
            print(f"OK {source_table} -> {target_table}: {written:,} rows; missing={len(missing)}, extra={len(extra)}")
        except Exception as exc:
            message = str(exc)[:2000]
            errors.append(f"{source_table}: {message}")
            failed += 1
            append_rows("monitoring.cfg_silver_load_control", [(RUN_ID, source_kind, source_table, target_table,
                contract_key[0] if contract_key else None, contract_key[1] if contract_key else None,
                "FAILED", 0, 0, 0, 0, started, datetime.utcnow(), message)], control_schema)

status = "FAILED" if errors else "SUCCESS"
error_text = " | ".join(errors)[:4000] if errors else None
error_sql = "NULL" if error_text is None else "'" + error_text.replace("'", "''") + "'"
spark.sql(f"""UPDATE monitoring.cfg_pipeline_run SET ended_at=current_timestamp(), status='{status}',
tables_succeeded={ok}, tables_failed={failed}, rows_read={total_read}, rows_written={total_written},
error_message={error_sql} WHERE run_id='{RUN_ID}'""")
if errors and FAIL_ON_TABLE_ERROR:
    raise RuntimeError(f"Silver formatting failed for {failed} table(s): {error_text}")
print(f"Silver run {RUN_ID}: {status}; {ok} tables and {total_written:,} rows")

StatementMeta(, 43152dc0-bcee-41bc-8411-b4bfc63e5f32, 18, Finished, Available, Finished, False)

OK bronze.additional_fee -> silver.slv_additional_fee: 257 rows; missing=1, extra=2
OK bronze.foster_home -> silver.slv_foster_home: 25 rows; missing=0, extra=1
OK bronze.foster_transport -> silver.slv_foster_transport: 25 rows; missing=0, extra=1
OK bronze.ipa_child -> silver.slv_ipa_child: 20 rows; missing=2, extra=1
OK bronze.provider_framework -> silver.slv_provider_framework: 291 rows; missing=0, extra=1
OK bronze.provider_home_age -> silver.slv_provider_home_age: 2,315 rows; missing=0, extra=1
OK bronze.provider_home_category -> silver.slv_provider_home_category: 2,384 rows; missing=0, extra=1
OK bronze.provider_home_spot_category -> silver.slv_provider_home_spot_category: 1,531 rows; missing=0, extra=1
OK bronze.referral_category -> silver.slv_referral_category: 478 rows; missing=0, extra=1
OK bronze.supervising_social_worker -> silver.slv_supervising_social_worker: 25 rows; missing=0, extra=1
OK archived.archived_additional_fee -> silver_archive.slv_additional_fee: 5,182 rows; 

RuntimeError: Silver formatting failed for 60 table(s): bronze.foster_carer: An error occurred while calling o18218.saveAsTable.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 3336.0 failed 4 times, most recent failure: Lost task 0.3 in stage 3336.0 (TID 23951) (vm-c8565992 executor 2): org.apache.spark.SparkException: [TASK_WRITE_FAILED] Task failed while writing rows to abfss://fefdb483-d26c-4bd9-9a4f-0c41cc786770@onelake.dfs.fabric.microsoft.com/d286fa39-f255-4ba7-a982-32cb69362ef7/Tables/silver/slv_foster_carer.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.taskFailedWhileWritingRowsError(QueryExecutionErrors.scala:777)
	at org.apache.spark.sql.delta.files.DeltaFileFormatWriter$.executeTask(DeltaFileFormatWriter.scala:621)
	at org.apache.spark.sql.delta.files.DeltaFileFormatWriter$.$anonfun$executeWrite$4(DeltaFileFormatWriter.scala:387)
	at org.apache.spark.scheduler.ResultTask.runTask(ResultTask.scala:93)
	at org.apache.spark.TaskCont

In [21]:
df = spark.sql("SELECT * FROM LH_BCT_WMPP.monitoring.cfg_schema_drift_event LIMIT 1000")
display(df)

StatementMeta(, 43152dc0-bcee-41bc-8411-b4bfc63e5f32, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 29af1a32-0a20-4b4a-b050-fcb61633a866)

In [22]:
df = spark.sql("SELECT * FROM LH_BCT_WMPP.monitoring.cfg_silver_load_control LIMIT 1000")
display(df)

StatementMeta(, 43152dc0-bcee-41bc-8411-b4bfc63e5f32, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e81f73cb-be22-4f29-a5c4-3222bac6533a)

In [ ]:
df = spark.sql("SELECT * FROM LH_BCT_WMPP.silver.slv_offer LIMIT 1000")
display(df)

StatementMeta(, 43152dc0-bcee-41bc-8411-b4bfc63e5f32, -1, Cancelled, , Cancelled, True)

In [20]:
df1=spark.table("bronze.additional_fee")
df2=spark.table("silver.slv_additional_fee")


print("df1")
print(df1.columns)

print("df2")
print(df2.columns)  


StatementMeta(, 43152dc0-bcee-41bc-8411-b4bfc63e5f32, 22, Finished, Available, Finished, False)

df1
['additional_fee_id', 'offer_id', 'fee_title', 'number_of_hours', 'other_fee_title', 'fee_frequency', 'rate', 'export_date']
df2
['additional_fee_id', 'offer_id', 'fee_code', 'number_of_hours', 'other_fee_title', 'fee_frequency', 'rate', '_record_source', '_source_table', '_source_file_path', '_export_date', '_silver_run_id', '_silver_load_ts']
